In [ ]:
import pandas as pd

pth_ref = os.path.join(pth_data, 'refs', 'AnimalTFDBv4')

# transcription factors
fn = os.path.join(pth_ref, 'Mus_musculus_TF')
tf = pd.read_csv(fn, sep = '\t').Symbol
tf = tf.loc[tf.isin(adata.var_names)].values
tf.size

1289

In [30]:
# # balanced sample
# n = adata.obs.source.value_counts().min()
# ix = (adata.obs.groupby('source', observed = True)
#       .sample(n = n, random_state = 1).index)
adata_sample = adata.copy(); adata_sample

AnnData object with n_obs × n_vars = 6405 × 14985
    obs: 'sample', 'celltype', 'source', 'early', 'late', 'spectrum', 't_source', 't', 'seg', 'edge', 'milestones'
    uns: 'source_colors', 'celltype_colors', 'graph', 'epg', 'pseudotime_list', 'milestones_colors', 'seg_colors'
    obsm: 'X_pca', 'X_spectrum', 'X_umap', 'X_umap_traj', 'X_R'

In [34]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# X (features): TF expression, source
src = adata_sample.obs.source.copy()
X = adata_sample[:, tf].X.toarray()
X = StandardScaler().fit_transform(X)
src = pd.get_dummies(src).values.astype(float)
X = np.concatenate((X, src), axis = 1)

# y (targets): t
y = adata_sample.obs.t_source.values.reshape(-1, 1)
y = StandardScaler().fit_transform(y).flatten()

# linear regression (y ~ X)
reg = LinearRegression(fit_intercept = False).fit(X, y)

# TF coefficients (beta)
beta = pd.Series(reg.coef_[:-src.shape[1]], index = tf)
beta = beta.sort_values(ascending = False); beta

Prrx1     0.125737
Dlx2      0.069751
Alx4      0.068982
Dlx1      0.056616
Meox1     0.052656
            ...   
T        -0.060460
En1      -0.077821
Otx2     -0.081503
Ybx1     -0.105955
Pou5f1   -0.141178
Length: 1289, dtype: float64